# Figure 7 CES Ward profiles

This is the original radar-chart code for panels a–d. The manuscript assembles the same four profiles into a different panel layout. The spatial map in panel e is not included.

Input files are referenced by name; place the existing inputs in `INPUT_DIR` or point it to their location. Only figure files are saved if the plotting cells are run.


In [ ]:
from pathlib import Path
import pandas as pd

# Point this to the existing input files for the selected panel.
INPUT_DIR = Path(".")
OUTPUT_DIR = Path("figure_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## Panels a–d Four standardised CES profiles

Source: `CES_EXP_Aggregation_and_Ward_Clustering.ipynb`, cells 17 and 45. The counts table is used to calculate positive proportions. The original Ward clustering, cluster order and plotting instructions are retained.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import math
from datetime import datetime

# Set matplotlib to use the installed Chinese font and handle minus signs
# The font 'WenQuanYi Zen Hei' was verified in the previous step.
plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei', 'DejaVu Sans', 'Arial Unicode MS'] # Use the verified font name
plt.rcParams['axes.unicode_minus'] = False

# Define the output directory and input file path
output_directory = str(OUTPUT_DIR)
input_csv_path = INPUT_DIR / "ces_category_sentiment_counts_table.csv"

# Ensure output directory exists
os.makedirs(output_directory, exist_ok=True)

# Load data, setting 'scenic_spot' as index
# Renaming 'df' to 'sentiment_counts_df' to avoid overwriting the main comments DataFrame
sentiment_counts_df = pd.read_csv(input_csv_path, index_col='scenic_spot')

# CES categories
categories = [
    "Aesthetic_values",
    "Cultural_heritage_values",
    "Inspiration",
    "Knowledge_and_education",
    "Recreation_and_visitation",
    "Sense_of_place",
    "Social_relations",
    "Spiritual_and_religious_value"
]

# Generate current timestamp for unique filenames
timestamp = datetime.now().strftime('%Y%m%d%H%M%S')

print("Generating visualization charts...")

# Helper function to clean category names for English display
def clean_category_name(name):
    return name.replace('_values', '').replace('_and', ' & ').replace('_', ' ').title()

cleaned_categories = [clean_category_name(cat) for cat in categories]

# --------------------------------------------------
# Plot 2: Normalized Positive CES Radar Charts
# --------------------------------------------------
ratio_data = pd.DataFrame()

# Calculate normalized positive CES ratio
for cat in categories:
    positive_col = f"{cat}_SNT_1"
    # Ensure the column exists and avoid division by zero
    if positive_col in sentiment_counts_df.columns and 'Total Comments' in sentiment_counts_df.columns:
        # Ensure 'Total Comments' is not zero to avoid division by zero error
        ratio_data[cat] = sentiment_counts_df[positive_col] / sentiment_counts_df["Total Comments"].replace(0, np.nan)
    else:
        ratio_data[cat] = 0.0 # Assign 0 if data is missing

# Drop rows where all values are NaN (if any scenic spots had no comments or no positive CES data)
ratio_data.dropna(how='all', inplace=True)

# Reindex ratio_data columns to match the 'categories' order for consistent plotting
ratio_data = ratio_data[categories]


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import pdist
from sklearn.preprocessing import StandardScaler
import os
import math
import seaborn as sns

# Ensure font settings are loaded for Chinese characters
plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei', 'DejaVu Sans', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

# Re-using output_directory from previous cells
output_directory = str(OUTPUT_DIR)
os.makedirs(output_directory, exist_ok=True)

print("--- 开始进行 Ward 层次聚类分析 (包含标准化和固定k=4) ---")

# User-provided mapping from park name to code
park_name_to_code_map = {
    "香山-植物园": "HG1",
    "圆明园": "HG2",
    "颐和园": "HG3",
    "八大处公园": "P1",
    "巴沟山水园": "P2",
    "百望山森林公园": "P3",
    "北京西山国家森林公园": "P4",
    "北坞公园": "P5",
    "畅春新园体育休闲广场": "P6",
    "茶棚公园": "P7",
    "国家植物园": "P8",
    "船营公园": "P9",
    "丹青圃公园": "P10",
    "东升八家郊野公园": "P11",
    "功德寺公园": "P12",
    "海淀公园": "P13",
    "荷清园": "P14",
    "六郎庄公园": "P15",
    "妙云寺公园": "P16",
    "南旱河公园": "P17",
    "上地公园": "P18",
    "石渠公园": "P19",
    "树村公园": "P20",
    "王庄公园": "P21",
    "万柳公园": "P22",
    "肖家河公园": "P23",
    "西水磨公园": "P24",
    "一亩园公园": "P25",
    "影湖楼公园": "P26",
    "玉东郊野公园": "P27",
    "中坞公园": "P28",
}

# Use `ratio_data` as the clustering input.
# Fill NaN with 0, assuming a missing value indicates no positive perception for that category.
ward_data = ratio_data.copy().fillna(0)

# --- 1. Standardise the data with StandardScaler ---
print("对数据进行标准化处理...")
scaler = StandardScaler()
scaled_ward_data = scaler.fit_transform(ward_data)
scaled_ward_df = pd.DataFrame(scaled_ward_data, index=ward_data.index, columns=ward_data.columns, dtype=float)

# --- 2. Calculate the distance matrix ---
# Calculate Euclidean distances on standardised data.
distance_matrix = pdist(scaled_ward_df, metric='euclidean')

# --- 3. Run Ward hierarchical clustering ---
linkage_matrix = linkage(distance_matrix, method='ward')

# --- 4. Set k = 4 and assign clusters ---
k = 4 # Set the requested number of clusters to 4.
print(f"根据用户要求，设定聚类数量 k = {k}...")
# fcluster partitions linkage_matrix at k; criterion='maxclust' makes k the maximum number of clusters.
cluster_assignments = fcluster(linkage_matrix, k, criterion='maxclust')

# Add the cluster assignments to the DataFrame.
# fcluster returns one-based IDs. Convert to zero-based IDs for consistency with Python and the KMeans results.
ward_clustering_results = ward_data.copy() # Use original data's index and structure for storing cluster ID
ward_clustering_results['cluster'] = cluster_assignments - 1 # Adjust to 0-indexed clusters

# --- 5. Describe the cluster profiles ---
# Average the original ratio_data by cluster for each CES category to retain interpretable radar-chart values on a 0–1 scale.
cluster_centers_for_plotting = scaled_ward_df.groupby(ward_clustering_results['cluster'])[categories].mean()

# Count the parks in each cluster.
cluster_counts = ward_clustering_results['cluster'].value_counts().sort_index()

# --- 6. Plot the cluster mean radar charts with individual park profiles ---
print("\n--- 生成每个群组的平均雷达图 (并叠加群组内公园个体雷达图) ---")

num_clusters = len(cluster_centers_for_plotting)
cols = 3 # Number of columns in the subplot grid.
rows = math.ceil(num_clusters / cols) # Number of rows in the subplot grid.
num_vars = len(categories)

# Calculate one radar chart angle per category.
angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
angles += angles[:1] # Close the radar chart polygon.

fig_clusters, axes_clusters = plt.subplots(rows, cols, figsize=(20, 6 * rows), subplot_kw=dict(polar=True))
axes_clusters = axes_clusters.flatten()

# Helper to format category names for English display.
def clean_category_name(name):
    return name.replace('_values', '').replace('_and', ' & ').replace('_', ' ').title()

cleaned_categories = [clean_category_name(cat) for cat in categories]
individual_park_palette = sns.color_palette('tab20', 20) # Colour palette for the individual parks in the cluster.

# Define the custom color palette for the cluster averages as requested
# The order of colors should correspond to the ordered_cluster_ids: 3, 2, 0, 1
custom_cluster_colors = {
    3: '#D2E7BE', # CES-P1
    2: '#8DB7B5', # CES-P2
    0: '#48869D', # CES-P3
    1: '#2E557E'  # CES-P4
}

# Determine dynamic y-axis limits for scaled data
global_min_val = min(scaled_ward_df.min().min(), cluster_centers_for_plotting.min().min())
global_max_val = max(scaled_ward_df.max().max(), cluster_centers_for_plotting.max().min())

# Ensure 0 is always visible and set symmetric limits if appropriate
y_lim_buffer = 0.1 # 10% buffer
if global_min_val < 0 and global_max_val > 0:
    # Symmetric range around 0
    y_max_abs = max(abs(global_min_val), abs(global_max_val)) * (1 + y_lim_buffer)
    y_lower_lim = -y_max_abs
    y_upper_lim = y_max_abs
    y_ticks = np.linspace(y_lower_lim, y_upper_lim, 5).round(1) # 5 ticks for reasonable density
elif global_min_val >= 0:
    # All positive
    y_lower_lim = 0
    y_upper_lim = global_max_val * (1 + y_lim_buffer)
    y_ticks = np.linspace(y_lower_lim, y_upper_lim, 5).round(1)
else: # global_max_val <= 0
    # All negative (unlikely for Z-scores, but for completeness)
    y_upper_lim = 0
    y_lower_lim = global_min_val * (1 + y_lim_buffer)
    y_ticks = np.linspace(y_lower_lim, y_upper_lim, 5).round(1)

# Ensure y_ticks are unique and sorted
y_ticks = np.unique(y_ticks).tolist()
y_ticks.sort()

# Mapping for new cluster names as requested
cluster_id_to_name = {
    3: 'CES-P1',
    2: 'CES-P2',
    0: 'CES-P3',
    1: 'CES-P4'
}

# Specify iteration order to display CES-P1, CES-P2, CES-P3 on top row, and CES-P4 on bottom left.
ordered_cluster_ids = [3, 2, 0, 1]

for i, cluster_id in enumerate(ordered_cluster_ids):
    center_values = cluster_centers_for_plotting.loc[cluster_id]
    ax_c = axes_clusters[i]
    values_cluster = center_values.values.tolist()
    values_cluster += values_cluster[:1] # Close the radar chart polygon.

    # Get the specific color for this cluster average from the custom_cluster_colors map
    cluster_color = custom_cluster_colors.get(cluster_id, '#000000') # Default to black if not found

    # Get the new cluster name
    new_cluster_name = cluster_id_to_name.get(cluster_id, f'Cluster {cluster_id}')

    # Draw the cluster mean with a thick solid line in the custom colour.
    ax_c.plot(angles, values_cluster, color=cluster_color, linewidth=3, linestyle='solid', label=f'{new_cluster_name} Average')
    ax_c.fill(angles, values_cluster, color=cluster_color, alpha=0.3)

    # List the parks assigned to this cluster.
    parks_in_cluster = ward_clustering_results[ward_clustering_results['cluster'] == cluster_id].index.tolist()

    # Draw a radar profile for each park in the cluster.
    for park_idx, park_name in enumerate(parks_in_cluster):
        park_row_data = scaled_ward_df.loc[park_name, categories] # Read each park profile from the original ward_data, a copy of ratio_data.
        values_park = park_row_data.values.tolist()
        values_park += values_park[:1] # Close the radar chart polygon.

        # Assign colours from the palette, cycling if there are more parks than colours.
        color = individual_park_palette[park_idx % len(individual_park_palette)]

        # Draw individual park profiles with thin dashed lines in different colours.
        # Add label using the mapped park code
        park_code = park_name_to_code_map.get(park_name, park_name) # Use code if available, else original name
        ax_c.plot(angles, values_park, color=color, linewidth=1.5, linestyle='--', alpha=0.8, label=park_code)

    ax_c.set_theta_offset(np.pi / 2)
    ax_c.set_theta_direction(-1)

    ax_c.set_xticks(angles[:-1])
    ax_c.set_xticklabels(cleaned_categories, fontsize=9, color='darkgreen')

    # Set dynamic y-ticks and limits for scaled data
    ax_c.set_ylim(y_lower_lim, y_upper_lim)
    ax_c.set_yticks(y_ticks)
    ax_c.set_yticklabels([f'{tick:.1f}' for tick in y_ticks], color="grey", size=8)

    # Include the cluster ID and number of parks in the title.
    ax_c.set_title(f'{new_cluster_name} (Parks: {cluster_counts.get(cluster_id, 0)})', size=14, color='black', y=1.15, fontweight='bold')

    # Add a legend including both the cluster average and individual parks
    # Using get_legend_handles_labels to avoid duplicate entries if any
    handles, labels = ax_c.get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    ax_c.legend(by_label.values(), by_label.keys(), loc='upper right', bbox_to_anchor=(1.5, 1.1), fontsize=8, frameon=False, title="Legend", ncol=2)

# Hide unused subplot axes.
for j in range(num_clusters, len(axes_clusters)):
    fig_clusters.delaxes(axes_clusters[j])

fig_clusters.tight_layout()
fig_clusters.subplots_adjust(wspace=0.5, hspace=0.7)

cluster_radar_path = os.path.join(output_directory, f"ces_ward_cluster_radar_charts_k{k}.png")
fig_clusters.savefig(cluster_radar_path, dpi=300, bbox_inches='tight')
plt.close(fig_clusters)
print(f"Ward 层次聚类群组平均雷达图（含个体公园）已保存: {cluster_radar_path}")

print("Ward 层次聚类和可视化完成。")
